# Ensemble Methods — Bagging, Boosting & Stacking

Comprehensive guide to ensemble learning techniques that combine multiple models for superior performance.

## Topics Covered
1. **Bagging** — Bootstrap Aggregating (Random Forest)
2. **Boosting** — AdaBoost, Gradient Boosting, XGBoost
3. **Stacking** — Meta-learner combining diverse models
4. **Voting** — Hard and Soft voting classifiers
5. Performance comparison on real dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Base models
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# Ensemble methods
from sklearn.ensemble import (
    BaggingClassifier, RandomForestClassifier,
    AdaBoostClassifier, GradientBoostingClassifier,
    VotingClassifier, StackingClassifier,
    ExtraTreesClassifier
)

from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

## Load Dataset

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Dataset: {data.DESCR.split(chr(10))[0]}")
print(f"Shape: {X.shape}, Classes: {np.bincount(y)}")

---
## 1. Bagging (Bootstrap Aggregating)

Trains multiple models on random subsets of the data (with replacement) and averages predictions.

**Key idea**: Reduce variance by averaging many high-variance models.

In [ ]:
# Bagging with Decision Trees as base
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    max_samples=0.8,
    max_features=0.8,
    bootstrap=True,
    oob_score=True,
    random_state=42,
    n_jobs=-1
)
bagging.fit(X_train_scaled, y_train)

print(f"Bagging Accuracy: {bagging.score(X_test_scaled, y_test):.4f}")
print(f"OOB Score:        {bagging.oob_score_:.4f}")

# Compare with single Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train_scaled, y_train)
print(f"\nSingle DT Acc:    {dt.score(X_test_scaled, y_test):.4f}")
print(f"Improvement:      +{(bagging.score(X_test_scaled, y_test) - dt.score(X_test_scaled, y_test))*100:.2f}%")

---
## 2. Boosting

Sequentially trains weak learners, each focusing on the mistakes of the previous one.

**Key idea**: Reduce bias by iteratively correcting errors.

### 2.1 AdaBoost

In [ ]:
ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),  # Decision stump
    n_estimators=200,
    learning_rate=0.5,
    random_state=42
)
ada.fit(X_train_scaled, y_train)
print(f"AdaBoost Accuracy: {ada.score(X_test_scaled, y_test):.4f}")

# Staged accuracy (performance vs n_estimators)
staged_scores = [score for score in ada.staged_score(X_test_scaled, y_test)]

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(staged_scores)+1), staged_scores, color='steelblue')
plt.xlabel('Number of Estimators')
plt.ylabel('Test Accuracy')
plt.title('AdaBoost: Accuracy vs Number of Weak Learners')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 2.2 Gradient Boosting

In [ ]:
gb = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,
    random_state=42
)
gb.fit(X_train_scaled, y_train)
print(f"Gradient Boosting Accuracy: {gb.score(X_test_scaled, y_test):.4f}")

# Feature importance
importances = pd.Series(gb.feature_importances_, index=data.feature_names)
top_features = importances.nlargest(10)

plt.figure(figsize=(10, 5))
top_features.sort_values().plot(kind='barh', color='coral')
plt.xlabel('Feature Importance')
plt.title('Gradient Boosting — Top 10 Features')
plt.tight_layout()
plt.show()

### 2.3 Learning Rate Effect

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
learning_rates = [0.01, 0.1, 1.0]

for ax, lr in zip(axes, learning_rates):
    gb_temp = GradientBoostingClassifier(
        n_estimators=300, learning_rate=lr,
        max_depth=3, random_state=42
    )
    gb_temp.fit(X_train_scaled, y_train)
    
    train_staged = [s for s in gb_temp.staged_score(X_train_scaled, y_train)]
    test_staged = [s for s in gb_temp.staged_score(X_test_scaled, y_test)]
    
    ax.plot(train_staged, label='Train', color='steelblue')
    ax.plot(test_staged, label='Test', color='coral')
    ax.set_title(f'Learning Rate = {lr}')
    ax.set_xlabel('Boosting Iterations')
    ax.set_ylabel('Accuracy')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('Effect of Learning Rate on Gradient Boosting', fontsize=14)
plt.tight_layout()
plt.show()

---
## 3. Voting Classifier

Combines predictions from multiple diverse models.

In [ ]:
# Hard Voting (majority vote)
voting_hard = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(max_iter=1000)),
        ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
        ('svc', SVC(kernel='rbf', random_state=42)),
        ('knn', KNeighborsClassifier(n_neighbors=5))
    ],
    voting='hard'
)

# Soft Voting (probability averaging)
voting_soft = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(max_iter=1000)),
        ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
        ('svc', SVC(kernel='rbf', probability=True, random_state=42)),
        ('knn', KNeighborsClassifier(n_neighbors=5))
    ],
    voting='soft'
)

voting_hard.fit(X_train_scaled, y_train)
voting_soft.fit(X_train_scaled, y_train)

print(f"Hard Voting Accuracy:  {voting_hard.score(X_test_scaled, y_test):.4f}")
print(f"Soft Voting Accuracy:  {voting_soft.score(X_test_scaled, y_test):.4f}")

# Individual model accuracies
print("\nIndividual Models:")
for name, model in voting_hard.named_estimators_.items():
    print(f"  {name}: {model.score(X_test_scaled, y_test):.4f}")

---
## 4. Stacking Classifier

Uses a meta-learner to combine base model predictions.

**Level 0**: Base models produce predictions  
**Level 1**: Meta-model learns from base model outputs

In [ ]:
stacking = StackingClassifier(
    estimators=[
        ('lr', LogisticRegression(max_iter=1000)),
        ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
        ('svc', SVC(kernel='rbf', probability=True, random_state=42)),
        ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42))
    ],
    final_estimator=LogisticRegression(max_iter=1000),  # Meta-learner
    cv=5,  # Cross-validation for generating meta-features
    n_jobs=-1
)

stacking.fit(X_train_scaled, y_train)
print(f"Stacking Accuracy: {stacking.score(X_test_scaled, y_test):.4f}")

---
## 5. Grand Comparison

In [ ]:
all_models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Bagging': BaggingClassifier(n_estimators=100, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Extra Trees': ExtraTreesClassifier(n_estimators=100, random_state=42),
    'AdaBoost': AdaBoostClassifier(n_estimators=200, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, random_state=42),
    'Hard Voting': voting_hard,
    'Soft Voting': voting_soft,
    'Stacking': stacking
}

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
results = {}

print(f"{'Model':<25} {'CV Mean':>10} {'CV Std':>10} {'Test Acc':>10}")
print("=" * 57)

for name, model in all_models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
    model.fit(X_train_scaled, y_train)
    test_acc = model.score(X_test_scaled, y_test)
    results[name] = scores
    print(f"{name:<25} {scores.mean():>10.4f} {scores.std():>10.4f} {test_acc:>10.4f}")

In [ ]:
# Visual comparison
plt.figure(figsize=(14, 6))
plt.boxplot(results.values(), labels=results.keys())
plt.xticks(rotation=30, ha='right')
plt.ylabel('Cross-Validation Accuracy')
plt.title('Ensemble Methods Comparison — Breast Cancer Dataset')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

| Method | Strategy | Reduces | Base Learners | Key Param |
|--------|----------|---------|---------------|----------|
| **Bagging** | Parallel, bootstrap | Variance | Homogeneous (trees) | n_estimators |
| **Random Forest** | Bagging + feature randomness | Variance | Decision Trees | max_features |
| **AdaBoost** | Sequential, reweighting | Bias | Weak learners (stumps) | learning_rate |
| **Gradient Boosting** | Sequential, residual fitting | Bias | Decision Trees | learning_rate, max_depth |
| **Voting** | Parallel, majority/probability | Variance | Heterogeneous | voting type |
| **Stacking** | Layered, meta-learner | Both | Heterogeneous | final_estimator |